## Portfolio Dataset Generation for Power BI

In [2]:
import pandas as pd
import numpy as np
import joblib

In [3]:
model_data = joblib.load("artifacts/model_data.joblib")

In [4]:
df_test_3 = pd.read_csv("artifacts/df_test_3.csv")

In [5]:
X_test_encoded= pd.read_csv("artifacts/X_test_encoded.csv")

In [6]:
y_test=pd.read_csv("artifacts/y_test.csv")

In [7]:
y_test.head()

,default
0,0
1,0
2,0
3,0
4,1


In [8]:
y_test=pd.read_csv("artifacts/y_test.csv").squeeze()

In [9]:
y_test.head()

0    0
1    0
2    0
3    0
4    1
Name: default, dtype: int64

In [10]:
model = model_data["model"]

In [11]:
predicted_default=model.predict(X_test_encoded)

In [12]:
probability_default=model.predict_proba(X_test_encoded)[:,1]

In [13]:
print(predicted_default[:10])

print(probability_default[:10])

[1 0 0 0 1 0 0 1 0 0]
[5.35282717e-01 1.33231950e-05 6.02080949e-03 6.98017516e-03
 9.21703142e-01 1.11200011e-06 8.28904653e-07 9.99928403e-01
 1.59441801e-04 2.05441661e-02]


### Calculation of Credit Score

In [14]:
credit_score = (300 + (1-probability_default)*600).astype(int)

In [15]:
credit_score[:10]

array([578, 899, 896, 895, 346, 899, 899, 300, 899, 887])

In [16]:
credit_score.max(),credit_score.min()

(899, 300)

### Risk Rating Creation

In [17]:
def risk_rating(score):
    if score<500:
        return 'Poor'
    elif score<650:
        return "Average"
    elif score <750:
        return "Good"
    else:
        return "Excellent"

In [18]:
rating_classification=[risk_rating(score) for score in credit_score]

#### Preparation of Dataset

In [19]:
powerbi_df=df_test_3.copy()

In [20]:
powerbi_df.head()

,age,gender,marital_status,employment_status,number_of_dependants,residence_type,years_at_current_address,city,state,zipcode,...,principal_outstanding,bank_balance_at_application,number_of_open_accounts,number_of_closed_accounts,enquiry_count,credit_utilization_ratio,default,loan_to_income,delinquency_ratio,avg_dpd_per_delinquency
0,36,M,Married,Self-Employed,3,Owned,24,Jaipur,Rajasthan,302001,...,1745379,806208,2,1,5,98,0,2.65,0.0,0.0
1,43,F,Single,Self-Employed,0,Owned,23,Delhi,Delhi,110001,...,1675530,1019647,4,0,5,32,0,1.24,0.0,0.0
2,30,M,Married,Self-Employed,4,Owned,27,Delhi,Delhi,110001,...,1613282,1000932,3,0,6,82,0,1.07,0.0,0.0
3,37,F,Single,Salaried,2,Owned,5,Pune,Maharashtra,411001,...,398614,167679,4,2,7,48,0,2.74,2.4,7.0
4,48,F,Single,Salaried,0,Mortgage,23,Chennai,Tamil Nadu,600001,...,383925,262180,3,1,8,97,1,2.04,10.7,6.4


In [23]:
powerbi_df['actual_default']=y_test
powerbi_df['predicted_default']=predicted_default
powerbi_df['probability_default']=probability_default
powerbi_df['risk_rating']=rating_classification
powerbi_df['credit_score']=credit_score


In [24]:
powerbi_df.head()

,age,gender,marital_status,employment_status,number_of_dependants,residence_type,years_at_current_address,city,state,zipcode,...,credit_utilization_ratio,default,loan_to_income,delinquency_ratio,avg_dpd_per_delinquency,actual_default,predicted_default,probability_default,risk_rating,credit_score
0,36,M,Married,Self-Employed,3,Owned,24,Jaipur,Rajasthan,302001,...,98,0,2.65,0.0,0.0,0,1,0.535283,Average,578
1,43,F,Single,Self-Employed,0,Owned,23,Delhi,Delhi,110001,...,32,0,1.24,0.0,0.0,0,0,0.000013,Excellent,899
2,30,M,Married,Self-Employed,4,Owned,27,Delhi,Delhi,110001,...,82,0,1.07,0.0,0.0,0,0,0.006021,Excellent,896
3,37,F,Single,Salaried,2,Owned,5,Pune,Maharashtra,411001,...,48,0,2.74,2.4,7.0,0,0,0.006980,Excellent,895
4,48,F,Single,Salaried,0,Mortgage,23,Chennai,Tamil Nadu,600001,...,97,1,2.04,10.7,6.4,1,1,0.921703,Poor,346


In [25]:
powerbi_df['actual_status']=powerbi_df['actual_default'].map({0:'Non-default',1:'Default'})

In [26]:
powerbi_df['predicted_status']=powerbi_df['predicted_default'].map({0:'Non-default',1:'Default'})

In [27]:
powerbi_df['probability%']=(powerbi_df['probability_default']*100).round(2)

### Decision for Recommendation

In [28]:
def approval_recommendation(score):
    if score >= 750:
        return "Approve"
    elif score >=650:
        return "Manual Reconsideration"
    else:
        return "Reject"

In [29]:
powerbi_df['recommendation']=powerbi_df['credit_score'].apply(approval_recommendation)

#### Income Range creation

In [30]:
powerbi_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12497 entries, 0 to 12496
Data columns (total 36 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   age                          12497 non-null  int64  
 1   gender                       12497 non-null  object 
 2   marital_status               12497 non-null  object 
 3   employment_status            12497 non-null  object 
 4   number_of_dependants         12497 non-null  int64  
 5   residence_type               12497 non-null  object 
 6   years_at_current_address     12497 non-null  int64  
 7   city                         12497 non-null  object 
 8   state                        12497 non-null  object 
 9   zipcode                      12497 non-null  int64  
 10  loan_purpose                 12497 non-null  object 
 11  loan_type                    12497 non-null  object 
 12  sanction_amount              12497 non-null  int64  
 13  processing_fee  

In [31]:
powerbi_df['loan_to_income_range']=pd.cut(powerbi_df['loan_to_income'],bins=[0, 2, 4, 6, float("inf")],
    labels=["Low", "Moderate", "High", "Very High"])

In [32]:
powerbi_df["credit_utilization_band"] = pd.cut(
    powerbi_df["credit_utilization_ratio"],
    bins=[0, 30, 60, 80, 100],
    labels=["Low", "Medium", "High", "Very High"],include_lowest=True
)

In [33]:
powerbi_df["Age Group"] = pd.cut(
    powerbi_df["age"],
    bins=[0,25,35,45,55,120],
    labels=["18-25","26-35","36-45","46-55","55+"],
    include_lowest=True
)

In [34]:
print(powerbi_df["credit_utilization_ratio"].describe())


count    12497.000000
mean        43.167800
std         29.246347
min          0.000000
25%         18.000000
50%         39.000000
75%         67.000000
max         99.000000
Name: credit_utilization_ratio, dtype: float64


In [35]:
powerbi_df.info()

powerbi_df.isnull().sum()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12497 entries, 0 to 12496
Data columns (total 39 columns):
 #   Column                       Non-Null Count  Dtype   
---  ------                       --------------  -----   
 0   age                          12497 non-null  int64   
 1   gender                       12497 non-null  object  
 2   marital_status               12497 non-null  object  
 3   employment_status            12497 non-null  object  
 4   number_of_dependants         12497 non-null  int64   
 5   residence_type               12497 non-null  object  
 6   years_at_current_address     12497 non-null  int64   
 7   city                         12497 non-null  object  
 8   state                        12497 non-null  object  
 9   zipcode                      12497 non-null  int64   
 10  loan_purpose                 12497 non-null  object  
 11  loan_type                    12497 non-null  object  
 12  sanction_amount              12497 non-null  int64   
 13  p

age                            0
gender                         0
marital_status                 0
employment_status              0
number_of_dependants           0
residence_type                 0
years_at_current_address       0
city                           0
state                          0
zipcode                        0
loan_purpose                   0
loan_type                      0
sanction_amount                0
processing_fee                 0
gst                            0
net_disbursement               0
loan_tenure_months             0
principal_outstanding          0
bank_balance_at_application    0
number_of_open_accounts        0
number_of_closed_accounts      0
enquiry_count                  0
credit_utilization_ratio       0
default                        0
loan_to_income                 0
delinquency_ratio              0
avg_dpd_per_delinquency        0
actual_default                 0
predicted_default              0
probability_default            0
risk_ratin

In [36]:
powerbi_df.describe()

,age,number_of_dependants,years_at_current_address,zipcode,sanction_amount,processing_fee,gst,net_disbursement,loan_tenure_months,principal_outstanding,...,credit_utilization_ratio,default,loan_to_income,delinquency_ratio,avg_dpd_per_delinquency,actual_default,predicted_default,probability_default,credit_score,probability%
count,12497.000000,12497.000000,12497.000000,12497.000000,1.249700e+04,12497.000000,1.249700e+04,1.249700e+04,12497.000000,1.249700e+04,...,12497.000000,12497.000000,12497.000000,12497.000000,12497.000000,12497.000000,12497.000000,1.249700e+04,12497.000000,12497.000000
mean,39.580059,1.924462,16.080899,419759.822117,4.720543e+06,80215.970233,7.219437e+05,3.208639e+06,25.879891,1.349039e+06,...,43.167800,0.085941,1.551274,10.607418,3.338441,0.085941,0.145875,1.552966e-01,806.115868,15.529445
std,9.816181,1.534356,8.943087,168845.353173,6.296359e+06,107826.439689,9.704380e+05,4.313058e+06,12.390961,1.217817e+06,...,29.246347,0.280288,0.965773,17.308141,2.896805,0.280288,0.352995,3.088751e-01,185.202616,30.887612
min,18.000000,0.000000,1.000000,110001.000000,7.100000e+04,1000.000000,9.000000e+03,4.000000e+04,6.000000,3.599900e+04,...,0.000000,0.000000,0.300000,0.000000,0.000000,0.000000,0.000000,1.642178e-09,300.000000,0.000000
25%,33.000000,0.000000,8.000000,302001.000000,1.164000e+06,19640.000000,1.767600e+05,7.856000e+05,16.000000,4.309610e+05,...,18.000000,0.000000,0.780000,0.000000,0.000000,0.000000,0.000000,1.509328e-05,849.000000,0.000000
50%,40.000000,2.000000,16.000000,411001.000000,2.687000e+06,45300.000000,4.077000e+05,1.812000e+06,24.000000,1.019408e+06,...,39.000000,0.000000,1.160000,4.200000,4.400000,0.000000,0.000000,8.371581e-04,899.000000,0.080000
75%,46.000000,3.000000,24.000000,560001.000000,5.148000e+06,91800.000000,8.262000e+05,3.672000e+06,35.000000,1.807199e+06,...,67.000000,0.000000,2.430000,13.400000,5.800000,0.000000,0.000000,8.337968e-02,899.000000,8.340000
max,70.000000,5.000000,31.000000,700001.000000,5.119800e+07,921720.000000,8.295480e+06,3.686880e+07,59.000000,5.000000e+06,...,99.000000,1.000000,4.590000,100.000000,10.000000,1.000000,1.000000,1.000000e+00,899.000000,100.000000


In [37]:
powerbi_df.to_csv("artifacts/final_credit_portfolio.csv",index=False)

In [38]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

In [42]:
model=model_data['model']
y_pred = model.predict(X_test_encoded)
probability_default = model.predict_proba(X_test_encoded)[:, 1]

In [43]:
accuracy = accuracy_score(y_test, y_pred)
precision = precision_score(y_test, y_pred)
recall = recall_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
roc_auc = roc_auc_score(y_test, probability_default)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)
print("ROC AUC  :", roc_auc)

Accuracy : 0.9303032727854685
Precision: 0.5556774547449259
Recall   : 0.9432029795158287
F1 Score : 0.699344149119779
ROC AUC  : 0.9836771217402378


In [44]:
metrics_df = pd.DataFrame({"Metric": ["Accuracy","Precision","Recall","F1 Score","ROC AUC"],
                           "Value": [accuracy,precision,recall,f1,roc_auc]})
metrics_df

,Metric,Value
0,Accuracy,0.930303
1,Precision,0.555677
2,Recall,0.943203
3,F1 Score,0.699344
4,ROC AUC,0.983677


In [45]:
metrics_df.to_csv("artifacts/model_metrics.csv",index=False)

In [46]:
from sklearn.metrics import confusion_matrix, classification_report

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=["Actual Non-Default", "Actual Default"],
    columns=["Predicted Non-Default", "Predicted Default"]
)

cm_df.to_csv("artifacts/confusion_matrix.csv")

# Classification Report
report_df = pd.DataFrame(
    classification_report(
        y_test,
        y_pred,
        output_dict=True
    )
).transpose()

report_df.to_csv("artifacts/classification_report.csv")